# HW 1: Discrete Choice Modeling

Analyze the data contained in the file “YogurtLong.dta.”

First use Stata (or another package) to estimate a multinomial logit brand choice model for this data.  Restrict all coefficients to be identical across choices, except for the intercept for each brand.  Then replicate the results using a pure matrix algebra coding in Matlab, R or comparable language. 

To tie in to future assignments, I suggest creating a master program that calls fminunc.m for maximization.  fminunc will maximize another function which will merely sum the log likelihoods over all N individuals. 

The other function (say Li.m) will create an Nx1 vector of log likelihoods.  Li.m should involve creating an NT by 1 vector of likelihoods for each observation.  Then, use a command such as grpstats.m to take the products within each i to get an N by 1 vector of individual likelihoods that will be output.  (This step is not necessary for this assignment but will be needed in future assignments which is why we are doing it here)

Once this is complete, try adding a right hand side variable to your brand utilities that is an indicator for whether that brand was the last brand purchased.  How do you interpret the results with this added variable?

Also, please come to class prepared to discuss a paper of your choice and the role the data and application plays in helping the paper make a substantive contribution.  If you prefer to be directed to a paper, consider one of the following: 
- Rust (1987)
- Bronnenberg, Dhar and Dube (2009)
- Hartmann and Klapper (2018)

## Results from STATA

```text
Conditional logit choice model                 Number of obs      =     63,920
Case ID variable: choice_id                    Number of cases    =      12784

Alternatives variable: brand                   Alts per case: min =          5
                                                              avg =        5.0
                                                              max =          5

                                                  Wald chi2(2)    =     650.74
Log likelihood = -8781.5944                       Prob > chi2     =     0.0000

------------------------------------------------------------------------------
           b | Coefficient  Std. err.      z    P>|z|     [95% conf. interval]
-------------+----------------------------------------------------------------
brand        |
           p |   -32.7738   1.379016   -23.77   0.000    -35.47662   -30.07097
           f |    .256184   .0876735     2.92   0.003     .0843471    .4280208
-------------+----------------------------------------------------------------
0            |  (base alternative)
-------------+----------------------------------------------------------------
1            |
       _cons |   .9346668   .1472778     6.35   0.000     .6460076    1.223326
-------------+----------------------------------------------------------------
2            |
       _cons |   .3404846   .1183367     2.88   0.004      .108549    .5724202
-------------+----------------------------------------------------------------
3            |
       _cons |  -3.322132   .1386069   -23.97   0.000    -3.593796   -3.050467
-------------+----------------------------------------------------------------
4            |
       _cons |  -.3185887   .1185656    -2.69   0.007     -.550973   -.0862043
------------------------------------------------------------------------------
```

## Python implementation

In [1]:
# Imports
import numpy as np
import pandas as pd
import cvxpy as cp
from scipy.optimize import minimize, root
from scipy.special import logsumexp
from scipy.stats import norm
from statsmodels.tools.numdiff import approx_hess

In [2]:
# Import data
df = pd.read_csv("../data/YogurtLong.txt", sep="\t")
print(df.head())

   hh  qty        exp  nopurch  b1  b2  b3  b4    p1    p2    p3    p4  f1  \
0   1    2  40.900002        0   0   0   0   1  0.11  0.08  0.06  0.08   0   
1   1    0   8.830000        1   0   0   0   0  0.11  0.09  0.05  0.08   0   
2   1    0   3.880000        1   0   0   0   0  0.13  0.09  0.05  0.08   0   
3   1    0   0.780000        1   0   0   0   0  0.13  0.09  0.05  0.08   0   
4   1    0  39.240002        1   0   0   0   0  0.13  0.09  0.05  0.08   0   

   f2  f3  f4  tripnum  
0   0   0   0        1  
1   0   0   0        2  
2   0   0   0        3  
3   0   0   0        4  
4   0   0   0        5  


In [3]:
# Convert df to long (each row is a hh-trip-brand choice occasion)
df_long = (
    pd.wide_to_long(
        df,
        stubnames=["b", "p", "f"],
        i=["hh", "tripnum"],
        j="brand",
        suffix=r"\d+"
    )
    .reset_index()
    .sort_values(["hh", "tripnum", "brand"])
)
df_long = pd.get_dummies(df_long, columns=["brand"], prefix="brand", dtype=int)

# Create prev_purch variable
df_long["brand_num"] = df_long[["brand_1", "brand_2", "brand_3", "brand_4"]].idxmax(axis=1).str.replace("brand_", "").astype(int)
trip_choice = (
    df_long.loc[df_long["b"] == 1, ["hh", "tripnum", "brand_num"]]
    .rename(columns={"brand_num": "chosen_brand"})
)
trip_hist = (
    df_long[["hh", "tripnum"]]
    .drop_duplicates()
    .sort_values(["hh", "tripnum"])
    .merge(trip_choice, on=["hh", "tripnum"], how="left")
)
trip_hist["prev_purch"] = (
    trip_hist.groupby("hh")["chosen_brand"]
    .transform(lambda x: x.ffill().shift(1))
    .fillna(0)
    .astype(int)
)
df_long = df_long.merge(
    trip_hist[["hh", "tripnum", "prev_purch"]],
    on=["hh", "tripnum"],
    how="left",
)
df_long["prev_purch"] = (df_long["prev_purch"] == df_long["brand_num"]).astype(int)

df_long.head()

,hh,tripnum,exp,nopurch,qty,b,p,f,brand_1,brand_2,brand_3,brand_4,brand_num,prev_purch
0,1,1,40.900002,0,2,0,0.11,0,1,0,0,0,1,0
1,1,1,40.900002,0,2,0,0.08,0,0,1,0,0,2,0
2,1,1,40.900002,0,2,0,0.06,0,0,0,1,0,3,0
3,1,1,40.900002,0,2,1,0.08,0,0,0,0,1,4,0
4,1,2,8.830000,1,0,0,0.11,0,1,0,0,0,1,0


### I. `cvxpy` implementation

Note that the log-likelihood is concave. Let $i = 1, \ldots H$ index households and $t = 1, \ldots, T_h$ index household $i$'s trips. Then, the log-likelihood of observing household $i$ choose brand $j$ in trip $t$ is
\begin{align*}
\log\left( \prod_{i=1}^H \prod_{t=1}^{T_h} \Pr(y = j \mid x\beta) \right) 
&= \log\left( \prod_{i=1}^H \prod_{t=1}^{T_h} \frac{\exp(\tilde{v}_{ij})}{1 + \sum_{k=1}^J \exp(\tilde{v}_{ik})} \right) \\
&= \sum_{i=1}^H \sum_{t=1}^{T_h} \left(\tilde{v}_{ij} - \log\left(1 + \sum_{k=1}^J \exp(\tilde{v}_{ik})\right) \right) \\
&= \sum_{i=1}^H \sum_{t=1}^{T_h} \left(\tilde{v}_{ij} - \log\left(\exp(0) + \sum_{k=1}^J \exp(\tilde{v}_{ik})\right) \right)
\end{align*}
As the log-sum-exp function is convex (Boyd and Vandenberghe) and $v_{ik}$ is an affine function of $\beta$, the summand is concave in $\beta$ and so the log-likehihood function is concave. We can proceed by using a convex optimization solver (i.e., `cvxpy`) to maximize a concave function.

In [4]:
# Form X, Y matrices
coef_names = ['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p']
X = df_long[coef_names].to_numpy()
Y = df_long[['b']].to_numpy()

# Compute v_ij's
n_alts = 4
n, k = X.shape
beta = cp.Variable((k, 1))
v_ij = X @ beta

V = cp.reshape(v_ij, (n // n_alts, n_alts), order='C')
V = cp.hstack([np.zeros((n // n_alts, 1)), V])  # add zero utility for outside option
chosen_v = Y.T @ v_ij

loglik = chosen_v - cp.sum(cp.log_sum_exp(V, axis=1))
problem = cp.Problem(cp.Maximize(loglik))
opt_ll = problem.solve()
opt_beta = beta.value.flatten()

In [5]:
# Print coefficient estimates
for name, coef in zip(coef_names, opt_beta):
    print(f"{name:<10} | {coef:>12.4f}")

brand_1    |       0.9347
brand_2    |       0.3405
brand_3    |      -3.3221
brand_4    |      -0.3186
f          |       0.2562
p          |     -32.7738


### II. Generic optimizer implementation

I suppose the log-likelihood might not always be concave for more complicated models, so it is probably best to implement a solver that doesn't rely on convexity of the objective.

In [6]:
def loglik(beta, X, Y, hh_ids, n_alts):
    """
    Computes negative log-likelihood for the dataset, assuming a conditional logit model with an outside option.

    Parameters
    ----------
    beta : array
        Coefficients to be estimated.
    X : array
        Matrix of covariates
    Y : array
        Vector of choices
    hh_ids : array
        Household ID corresponding to each row of X and Y.
    n_alts : int
        Number of alternatives.

    Returns
    -------
    float
        Negative log-likelihood.
    """
    X = np.asarray(X)
    Y = np.asarray(Y).reshape(-1)
    hh_ids = np.asarray(hh_ids).reshape(-1)
    beta = np.asarray(beta).reshape(-1)

    n, k = X.shape
    n_choices = n // n_alts                         # number of hh-trip choice occasions

    v_ij = X @ beta                                 # utility for each hh-trip-brand
    V = v_ij.reshape(n_choices, n_alts, order="C")  # reshape to (choice occasion, alternative)
    chosen_v = np.multiply(Y, v_ij).\
        reshape(n_choices, n_alts).sum(axis=1)      # chosen utility per choice occasion
    V_with_outside = np.hstack([np.zeros((n_choices, 1)), V])   # add outside option (zero utility)
    log_denom = logsumexp(V_with_outside, axis=1)

    loglik_trip = chosen_v - log_denom                          # trip-level log-likelihoods
    _, hh_index = np.unique(hh_ids, return_inverse=True)        # group by household
    loglik_hh = np.bincount(hh_index, weights=loglik_trip)

    return -np.sum(loglik_hh)


def score_matrix(beta, X, Y, n_alts):
    """
    Returns choice-occasion score vectors s_t(beta) = d log L_t / d beta.
    Shape: (n_choices, k).
    """
    X = np.asarray(X)
    Y = np.asarray(Y).reshape(-1)
    beta = np.asarray(beta).reshape(-1)

    n, k = X.shape
    n_choices = n // n_alts

    # Reshape long format into (choice occasion, alternative, regressor)
    X3 = X.reshape(n_choices, n_alts, k, order="C")
    Y2 = Y.reshape(n_choices, n_alts, order="C")

    # Softmax probabilities including outside option with normalized utility 0
    V = X3 @ beta                                      # (n_choices, n_alts)
    V_with_outside = np.hstack([np.zeros((n_choices, 1)), V])
    P_with_outside = np.exp(V_with_outside - logsumexp(V_with_outside, axis=1, keepdims=True))
    P = P_with_outside[:, 1:]                          # drop outside option prob

    # Score per choice occasion: sum_j (y_tj - p_tj) x_tj
    S = np.einsum("ta,tak->tk", (Y2 - P), X3)
    return S


def mle_estimator(df, 
                  choice_var,
                  covariate_vars,
                  individual_ids,
                  n_alts,
                  beta_init=None,
                  ci_alpha=0.05,
                  robust_se=False,
                  method='BFGS'):
    """
    Estimates the multinomial logit model using maximum likelihood estimation.
    
    Parameters    
    ----------
    df : DataFrame
        The dataset to be used for estimation.
    choice_var : str
        The column name of the choice variable.
    covariate_vars : list
        A list of column names for the covariates.
    individual_ids : array
        An array of individual IDs.
    n_alts : int
        Number of alternatives.
    beta_init : array, optional
        Initial guess for coefficients. If None, defaults to zeros.
    ci_alpha : float
        Significance level for confidence intervals.
    robust_se : bool
        Whether to compute robust standard errors.
    method : str
        Optimization method.

    Returns    
    -------
    Dictionary containing:
        - optimized log-likelihood 
        - coefficient estimates
        - standard errors
        - confidence intervals for coefficients
    """
    # Prepare data
    Y = df[[choice_var]].to_numpy()
    X = df[covariate_vars].to_numpy()

    if beta_init is None:
        beta_init = np.zeros(X.shape[1])

    result = minimize(
        loglik, 
        beta_init, 
        args=(X, Y, individual_ids, n_alts),
        method=method,
    )

    opt_ll = -result.fun
    opt_beta = result.x

    # Standard errors and confidence intervals (using Hessian, under correct specification)
    H = approx_hess(opt_beta, loglik, args=(X, Y, individual_ids, n_alts))  # numerical Hessian at opt_beta
    H_inv = np.linalg.inv(H)
    se_beta = np.sqrt(np.diag(H_inv))
    z_score = norm.ppf(1 - ci_alpha / 2)
    ci = [(coef - z_score * se, coef + z_score * se) for coef, se in zip(opt_beta, se_beta)]

    output = {
        'opt_ll': opt_ll,
        'opt_beta': opt_beta,
        'se_beta': se_beta,
        'ci': ci,
    }

    if robust_se:
        # J = E[s_t s_t'] estimated by sample average of outer products of scores.
        S = score_matrix(opt_beta, X, Y, n_alts)       # (n_choices, k)
        n_choices = S.shape[0]
        J = S.T @ S

        # Sandwich variance: H^{-1} J H^{-1}
        V_robust = H_inv @ J @ H_inv
        se_beta_r = np.sqrt(np.diag(V_robust))
        ci_r = [(coef - z_score * se, coef + z_score * se) for coef, se in zip(opt_beta, se_beta_r)]

        output['se_beta'] = se_beta_r
        output['ci'] = ci_r
    
    # Print coefficient estimates with aligned decimals and grouped CI header
    widths = (12, 10, 10, 10, 10)

    header_top = f"{'Coefficient':^{widths[0]}} | {'Estimate':^{widths[1]}} | {'Std. Err.':^{widths[2]}} | {'Confidence Interval':^{widths[3] + widths[4] + 3}}"
    divider = "-+-".join("-" * w for w in widths)
    print(header_top)
    print(divider)

    for i, (name, coef) in enumerate(zip(covariate_vars, opt_beta)):
        row = " | ".join([
            f"{name:<{widths[0]}}",
            f"{coef:>{widths[1]}.6f}",
            f"{output['se_beta'][i]:>{widths[2]}.7f}",
            f"{output['ci'][i][0]:>{widths[3]}.5f}",
            f"{output['ci'][i][1]:>{widths[4]}.5f}"
        ])
        print(row)

    return output

In [7]:
hh_ids = df[['hh']].to_numpy()
results = mle_estimator(df_long, 
                        choice_var = 'b', 
                        covariate_vars = ['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p'], 
                        individual_ids = hh_ids, 
                        n_alts = 4, 
                        robust_se = False)

Coefficient  |  Estimate  | Std. Err.  |   Confidence Interval  
-------------+------------+------------+------------+-----------
brand_1      |   0.934656 |  0.1472744 |    0.64600 |    1.22331
brand_2      |   0.340475 |  0.1183336 |    0.10855 |    0.57241
brand_3      |  -3.322133 |  0.1386057 |   -3.59379 |   -3.05047
brand_4      |  -0.318598 |  0.1185625 |   -0.55098 |   -0.08622
f            |   0.256186 |  0.0876728 |    0.08435 |    0.42802
p            | -32.773687 |  1.3789813 |  -35.47644 |  -30.07093


### III. GMM Implementation

In [8]:
def moment_conditions(beta, X, Y, n_alts):
    """
    Trip-level moment conditions g_t(beta) for MNL.
    g_t(beta) = sum_j (y_tj - p_tj(beta)) * x_tj
    """
    X = np.asarray(X)
    Y = np.asarray(Y).reshape(-1)
    beta = np.asarray(beta).reshape(-1)

    n, k = X.shape
    n_choices = n // n_alts

    X3 = X.reshape(n_choices, n_alts, k, order="C")
    Y2 = Y.reshape(n_choices, n_alts, order="C")

    V = X3 @ beta
    V_with_outside = np.hstack([np.zeros((n_choices, 1)), V])
    P_with_outside = np.exp(V_with_outside - logsumexp(V_with_outside, axis=1, keepdims=True))
    P = P_with_outside[:, 1:]

    g = np.einsum("ta,tak->tk", (Y2 - P), X3)
    return g  # (n_choices, k)


def household_moment_matrix(beta, X, Y, hh_ids_trip, n_alts):
    """
    Household-level moments g_i(beta) = sum_{t in i} g_t(beta).

    Returns
    -------
    G_i : array, shape (n_households, k)
    hh_unique : array of household IDs in row order of G_i
    """
    g_t = moment_conditions(beta, X, Y, n_alts)  # (n_trips, k)
    hh_ids_trip = np.asarray(hh_ids_trip).reshape(-1)

    hh_unique, hh_index = np.unique(hh_ids_trip, return_inverse=True)
    n_households = len(hh_unique)
    k = g_t.shape[1]

    G_i = np.zeros((n_households, k))
    np.add.at(G_i, hh_index, g_t)
    return G_i, hh_unique


def gmm_objective(beta, X, Y, hh_ids_trip, n_alts, W):
    """
    Household-aggregated GMM criterion: Q(beta) = g_bar(beta)' W g_bar(beta).
    """
    G_i, _ = household_moment_matrix(beta, X, Y, hh_ids_trip, n_alts)
    g_bar = G_i.mean(axis=0)
    return g_bar @ W @ g_bar


def gmm_estimator(df,
                  choice_var,
                  covariate_vars,
                  individual_ids,
                  n_alts,
                  beta_init=None,
                  ci_alpha=0.05,
                  two_step=True,
                  method='BFGS'):
    """
    Estimates conditional logit with household-aggregated GMM moments.

    Moments are first summed over trips within household, then averaged across
    households. This yields household-clustered standard errors.
    """
    Y = df[[choice_var]].to_numpy()
    X = df[covariate_vars].to_numpy()
    k = X.shape[1]
    n_trips = X.shape[0] // n_alts

    if beta_init is None:
        beta_init = np.zeros(k)

    hh_ids = np.asarray(individual_ids).reshape(-1)
    if hh_ids.size == n_trips:
        hh_ids_trip = hh_ids
    elif hh_ids.size == X.shape[0]:
        hh_grid = hh_ids.reshape(n_trips, n_alts)
        if not np.all(hh_grid == hh_grid[:, [0]]):
            raise ValueError("individual_ids at long level must be constant within each choice occasion")
        hh_ids_trip = hh_grid[:, 0]
    else:
        raise ValueError("individual_ids must have length n_trips or n_trips * n_alts")

    def g_bar_fn(b):
        G_i, _ = household_moment_matrix(b, X, Y, hh_ids_trip, n_alts)
        return G_i.mean(axis=0)

    root_res = root(g_bar_fn, beta_init, method='hybr')
    beta_hat = root_res.x

    if (not root_res.success) or (np.linalg.norm(g_bar_fn(beta_hat)) > 1e-6):
        W = np.eye(k)
        min_res = minimize(
            gmm_objective,
            beta_init,
            args=(X, Y, hh_ids_trip, n_alts, W),
            method=method,
            options={'gtol': 1e-12, 'maxiter': 5000},
        )
        beta_hat = min_res.x
    else:
        min_res = None

    W = np.eye(k)

    if two_step:
        G_i_1, hh_unique = household_moment_matrix(beta_hat, X, Y, hh_ids_trip, n_alts)
        n_households = G_i_1.shape[0]
        S_hat = (G_i_1.T @ G_i_1) / n_households
        ridge = 1e-8    # helps with numerical stability
        W = np.linalg.inv(S_hat + ridge * np.eye(k))

        min_res_2 = minimize(
            gmm_objective,
            beta_hat,
            args=(X, Y, hh_ids_trip, n_alts, W),
            method=method,
            options={'gtol': 1e-12, 'maxiter': 5000},
        )
        beta_hat = min_res_2.x

    eps = 1e-6
    G = np.zeros((k, k))
    for j in range(k):
        e = np.zeros(k)
        e[j] = eps
        G[:, j] = (g_bar_fn(beta_hat + e) - g_bar_fn(beta_hat - e)) / (2 * eps)

    # Sandwich variance: (G'WG)^{-1} G' W S W G (G'WG)^{-1} / T
    G_i, hh_unique = household_moment_matrix(beta_hat, X, Y, hh_ids_trip, n_alts)
    n_households = G_i.shape[0]
    S = (G_i.T @ G_i) / n_households
    GWG = G.T @ W @ G
    GWG_inv = np.linalg.inv(GWG)
    V = GWG_inv @ (G.T @ W @ S @ W @ G) @ GWG_inv / n_households
    se_beta = np.sqrt(np.diag(V))

    z = norm.ppf(1 - ci_alpha / 2)
    ci = [(b - z * s, b + z * s) for b, s in zip(beta_hat, se_beta)]

    output = {
        'opt_beta': beta_hat,
        'se_beta': se_beta,
        'ci': ci,
        'W': W,
        'g_bar': g_bar_fn(beta_hat),
        'n_clusters': n_households,
    }

    widths = (12, 10, 10, 10, 10)
    header_top = (f"{'Coefficient':^{widths[0]}} | {'Estimate':^{widths[1]}} | "
                  f"{'Std. Err.':^{widths[2]}} | {'Confidence Interval':^{widths[3] + widths[4] + 3}}")
    divider = "-+-".join("-" * w for w in widths)
    print(header_top)
    print(divider)

    for i, (name, coef) in enumerate(zip(covariate_vars, beta_hat)):
        row = " | ".join([
            f"{name:<{widths[0]}}",
            f"{coef:>{widths[1]}.6f}",
            f"{se_beta[i]:>{widths[2]}.7f}",
            f"{ci[i][0]:>{widths[3]}.5f}",
            f"{ci[i][1]:>{widths[4]}.5f}"
        ])
        print(row)

    return output

In [9]:
hh_ids = df[['hh']].to_numpy()
results_gmm = gmm_estimator(df_long,
                             choice_var='b',
                             covariate_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p'],
                             individual_ids=hh_ids,
                             n_alts=4,
                             two_step=True)

Coefficient  |  Estimate  | Std. Err.  |   Confidence Interval  
-------------+------------+------------+------------+-----------
brand_1      |   0.934667 |  0.3256902 |    0.29633 |    1.57301
brand_2      |   0.340485 |  0.3539795 |   -0.35330 |    1.03427
brand_3      |  -3.322132 |  0.3212057 |   -3.95168 |   -2.69258
brand_4      |  -0.318589 |  0.3567979 |   -1.01790 |    0.38072
f            |   0.256184 |  0.1280672 |    0.00518 |    0.50719
p            | -32.773795 |  3.4634668 |  -39.56207 |  -25.98552


In [10]:
hh_ids = df[['hh']].to_numpy()
results_gmm = gmm_estimator(df_long,
                             choice_var='b',
                             covariate_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p'],
                             individual_ids=hh_ids,
                             n_alts=4,
                             two_step=False)

Coefficient  |  Estimate  | Std. Err.  |   Confidence Interval  
-------------+------------+------------+------------+-----------
brand_1      |   0.934667 |  0.3256902 |    0.29633 |    1.57301
brand_2      |   0.340485 |  0.3539795 |   -0.35330 |    1.03427
brand_3      |  -3.322132 |  0.3212057 |   -3.95168 |   -2.69258
brand_4      |  -0.318589 |  0.3567979 |   -1.01790 |    0.38072
f            |   0.256184 |  0.1280672 |    0.00518 |    0.50719
p            | -32.773795 |  3.4634668 |  -39.56207 |  -25.98552


## Adding brand inertia

The coefficient on `prev_purch` is 3.1820 and statistically significant, suggesting that previously purchasing a brand is associated with a 3.182 increase in the log-odds ($\exp(3.182) = 24.095$ in the odds ratio) of re-purchasing that brand (relative to the outside option) in the next trip. One could take this result to suggest that there is brand inertia.

In [11]:
hh_ids = df[['hh']].to_numpy()
results = mle_estimator(df_long, 
                        choice_var = 'b', 
                        covariate_vars = ['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
                        individual_ids = hh_ids, 
                        n_alts = 4, 
                        robust_se = True)

Coefficient  |  Estimate  | Std. Err.  |   Confidence Interval  
-------------+------------+------------+------------+-----------
brand_1      |  -0.699019 |  0.1843640 |   -1.06037 |   -0.33767
brand_2      |  -1.392080 |  0.1542433 |   -1.69439 |   -1.08977
brand_3      |  -3.825416 |  0.1471794 |   -4.11388 |   -3.53695
brand_4      |  -1.254406 |  0.1385400 |   -1.52594 |   -0.98287
f            |   0.349683 |  0.1007760 |    0.15217 |    0.54720
p            | -32.622003 |  1.6127586 |  -35.78295 |  -29.46105
prev_purch   |   2.466313 |  0.0599415 |    2.34883 |    2.58380


In [12]:
hh_ids = df[['hh']].to_numpy()
results = gmm_estimator(df_long, 
                        choice_var = 'b', 
                        covariate_vars = ['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
                        individual_ids = hh_ids, 
                        n_alts = 4, 
                        two_step = True)

Coefficient  |  Estimate  | Std. Err.  |   Confidence Interval  
-------------+------------+------------+------------+-----------
brand_1      |  -0.699012 |  0.4226015 |   -1.52730 |    0.12927
brand_2      |  -1.392075 |  0.4172538 |   -2.20988 |   -0.57427
brand_3      |  -3.825409 |  0.3230481 |   -4.45857 |   -3.19225
brand_4      |  -1.254401 |  0.3041322 |   -1.85049 |   -0.65831
f            |   0.349679 |  0.1256581 |    0.10339 |    0.59596
p            | -32.622074 |  3.6696694 |  -39.81449 |  -25.42965
prev_purch   |   2.466313 |  0.1843994 |    2.10490 |    2.82773


(The standard errors from GMM are clustered at the household level, hence the difference from standard errors produced by the MLE estimator. The GMM standard errors match those in the homogeneous MNL of `2-discrete-heterogeneity.ipynb`, which implemented robust standard errors clustered at the household level.)